In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
TRAINING_DATA = "training_data_clean.csv"


def prep_data():
    df = pd.read_csv(TRAINING_DATA)

    # rename
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
        ]


<bound method Series.unique of 0       3 — Neutral / Unsure
1                 4 — Likely
2       3 — Neutral / Unsure
3            5 — Very likely
4                 4 — Likely
               ...          
820     3 — Neutral / Unsure
821             2 — Unlikely
822          5 — Very likely
823             2 — Unlikely
824    1 — Not at all likely
Name: acad_tasks_rating, Length: 825, dtype: object>


In [ ]:
import numpy as np
import re
from sklearn.compose import ColumnTransformer
 # TODO: can use KNN to find the imputation point instead, in-person imputation
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import KNNImputer



# divide columns by type
text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

ord_categories = [
                '1 — Not at all likely',
                '2 — Unlikely',
                '3 — Neutral / Unsure',
                '4 — Likely',
                '5 — Very likely'
                ]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

def preprocess_text():
    return make_pipeline(
        SimpleImputer(strategy="constant", fill_value ="", missing_values=np.nan)
        # text embedding -> always use llm, tokenize and get embedding vector, using hugging face library (Ex. BERT)
        # return torch tensor, embedding is one of the properties

        # TFIDF manually (classical)
    )


def preprocess_ord():

    return make_pipeline(
        OrdinalEncoder(categories= [ord_categories] * len(ord_cols), handle_unknown='use_encoded_value', unknown_value=np.nan),
        IterativeImputer(
            max_iter=20,
            sample_posterior=True
        )

    )

def preprocess_cat():
    pass
    # TODO: research how to preprocess these

def create_preprocessor(df):
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    transformers = [
        ("preprocessing_pipeline_for_free_response_features", preprocess_text(), text_cols),
        ("preprocessing_pipeline_for_ordinal_rating_features", preprocess_ord(), ord_cols),
         ("preprocessing_pipeline_for_categorical_select_features", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


column_transformer = create_preprocessor()
column_transformer.set_output(transform="pandas")
# preprocessed_num_cat_features_df = column_transformer.fit_transform(
#     train_df[[NUMERICAL_FEATURE, CATEGORICAL_FEATURE]]
# )

In [ ]:

def split_data(df):
    students = df['id'].unique()
    train_df, test_df = train_test_split(students, test_size=0.3, random_state=42)
